# 03 — Run timing and PQC detection through PLEB

**You will:** write the detection profile, run the pipeline, find its branch-specific
QC tables and prove detection did not change the selected TOAs.
The supplied model is suitable for the artificial experiment; a wrong model can make
good measurements look like outliers. Check the fit before trusting a detector.

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, tarfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tomlkit
COURSE = Path(os.environ['PLEB_COURSE_HOME'])
sys.path.insert(0, str(COURSE))
import release_tools as lab
ROOT = lab.environment(Path(os.environ['PLEB_LEARNING_OUTPUT']))
REPO = ROOT/'dataset-repo'
CONFIGS = ROOT/'configs'
PULSARS = lab.PULSARS
CLI = [sys.executable, '-m', 'pleb.cli']

In [ ]:
common = json.loads((CONFIGS/'common.json').read_text())
detection = dict(common, branches=['prepared'], reference_branch='prepared',
    outdir_name='03_detect', run_tempo2=True, run_pqc=True,
    pqc_backend_col='sys', pqc_merge_tol_seconds=0.01, pqc_drop_unmatched=False,
    pqc_robust_enabled=True, pqc_robust_scope='backend', pqc_robust_z_thresh=6.0,
    pqc_fdr_q=0.01, pqc_mark_only_worst_per_day=False,
    pqc_add_orbital_phase=False, pqc_add_solar_elongation=False,
    pqc_structure_mode='none', pqc_step_enabled=False, pqc_dm_step_enabled=False,
    pqc_solar_events_enabled=False, pqc_orbital_phase_cut_enabled=False,
    pqc_eclipse_events_enabled=False, pqc_gaussian_bump_enabled=False,
    pqc_glitch_enabled=False, pqc_delta_chi2_thresh=1e12)
path = lab.write_config(CONFIGS/'03_detect.toml',detection)
commit_before = lab.command(['git','-C',REPO,'rev-parse','prepared']).strip()
lab.command(CLI+['--config',path], log=ROOT/'logs/03_detect.log')
assert lab.command(['git','-C',REPO,'rev-parse','prepared']).strip()==commit_before

## Read the statistical output in context

`pqc_backend_col="sys"` defines the comparison groups. Robust-z uses a median/MAD scale
and can highlight isolated deviations without a Gaussian standard deviation being dominated
by one extreme point. `pqc_fdr_q` concerns the relevant multiple-test decision in PQC; it
does not set a posterior probability that each flagged TOA is bad. Event membership has
a different interpretation from isolated outlier evidence.

We disable physical event families for this white-noise simulation. Those settings are
not a general release policy. Real eclipses, solar effects and coherent structures must
be explicitly modelled/reviewed before deciding which observations to comment.

In [ ]:
qc_paths = {}
for psr in PULSARS:
    matches = list((ROOT/'results/03_detect').rglob(f'{psr}_qc.csv'))
    assert len(matches)==1, (psr,matches)
    qc_paths[psr] = str(matches[0])
    frame = pd.read_csv(matches[0])
    print(psr, 'QC rows:',len(frame),'columns:',list(frame.columns))
    assert len(frame)==288, 'Check whether a fit or QC subprocess failed or omitted TOAs'
    display(frame.head())
(ROOT/'qc_paths.json').write_text(json.dumps(qc_paths,indent=2))
print('Full logs:', ROOT/'logs/03_detect.log')

**Your task:** locate the Tempo2 logs and PQC tables yourself. Explain how `prepared`
in the results path differs from the stage name `03_detect`. Identify detector columns
that are evidence and columns that identify a measurement. Next: review those measurements.